In [ ]:
#FORWARD CHAINING ALGORITHM

#import tools
import csv #reading required file
import random #to shuffle cars before splitting into training and test data
from collections import Counter #counts (like cars per class)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix #scoring tools 

#no header row in car.data so columns named manually - rows become dictionaries
columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
with open("car.data") as f:
    rows = [dict(zip(columns, line)) for line in csv.reader(f) if line] #if line skips blanks

random.seed(71)#fixed seed so same split each run, results thereby reproducible. shuffling both groups gets a mix of classes
shuffled = rows[:] #copy so OG order remains
random.shuffle(shuffled) #mixy mix cars groups to get fair combination.else test set would only have a certain group of cars since car.data is sorted

cut = int(0.7 * len(shuffled)) #70% of 1728, 1209 = training data to design rules. the 519 test cars used to score the rules
train, test = shuffled[:cut], shuffled[cut:] #train = design rues. test = score rules. 70% train, 30% test

#rules:
    #set of conditions, conclusion and fires when all conditions = known facts. 
rules = [
    ({"safety=low"}, "class=unacc"), #not family safe or safe for daily usage
    ({"persons=2"}, "class=unacc"), #not gonna hold a family
    ({"buying=vhigh", "maint=vhigh"}, "class=unacc"),#expensive to buy AND run
    ({"buying=vhigh", "maint=high"}, "class=unacc"), # high buying ALONE isn't grounds for rejection
    ({"buying=high", "maint=vhigh"}, "class=unacc"),

    #vgood rules - ideal family car is safe with big boot and affordable
    ({"buying=low", "safety=high", "lug_boot=big"}, "class=vgood"), 
    ({"buying=med", "maint=med", "safety=high", "lug_boot=big"}, "class=vgood"),
    ({"buying=med", "maint=low", "safety=high", "lug_boot=big"}, "class=vgood"),

    #cheap buying and running: good
    ({"buying=low", "maint=low"}, "class=good"), 
    ({"buying=low", "maint=med"}, "class=good"),
    ({"buying=med", "maint=low"}, "class=good"),

    #reasoably safe with boot space. last since they're more broad. acc
    ({"safety=high"}, "class=acc"),
    ({"safety=med", "lug_boot=big"}, "class=acc"),
    ({"safety=med", "lug_boot=med"}, "class=acc")
] # no rule fires meaning the default response is class=unacc which is set in a later predict function

#start from known facts. fire until conditions known + conclusion added as new fact
def forward_chain(facts, rules):
    known = set(facts) #a copy so callers facts aren't changed
    changed = True #loop runs AT LEAST once
    while changed: #loop until nothing else derived
        changed = False #assumes nothing new has been added this pass around
        for conditions, conclusion in rules:
            if conditions <= known and conclusion not in known: #conditions <= known: are all conditions known facts? subset test
                #not in known stops adding the same fact twice
                known.add(conclusion) #rule fires and adds conclusion
                changed = True #loop again if something new opops ip
    return known #starting facts + all derived

#same as above but records trace of fired rules. show why answer was derived
def forward_chain_trace(facts, rules):
    known, trace = set(facts), [] #trace starts as empty list
    changed = True
    while changed:
        changed = False
        for conditions, conclusion in rules:
            if conditions <= known and conclusion not in known:
                known.add(conclusion)
                trace.append((sorted(conditions), conclusion)) #clock why broski
                changed = True
    return known, trace #facts and list of rules fired


row = {"buying": "low", "safety": "high", "persons": "4", "lug_boot": "big"}
#this above is test to show how fwd chain work. dictionary of attributes/values

facts = {f"{attribute}={value}" for attribute, value in row.items()} #attributes=value pair turns into fact )e.g. buying=high

#gna run forward chaining and print just the facts but can derive more than one class. conflict :(
derived = forward_chain(facts, rules) 
print(sorted(f for f in derived if f.startswith("class=")))

#show which rules fired, wat conclusion was
known, trace = forward_chain_trace(facts, rules)
for conditions, conclusion in trace:
    print(f"{' + '.join(conditions)} --> {conclusion}") #trace of fired rules + conclusion. how conclusion was derived

#fwdch derives conclusions from first match conflict res. 
def predict(row, rules, default="class=unacc"): 
    facts = {f"{a}={v}" for a, v in row.items() if a != "class"} 
    derived = forward_chain(facts, rules) #fwd chain derives each possible conclusion
    for conditions, conclusion in rules:  #first priority order
        if conclusion.startswith("class=") and conditions <= derived:
            return conclusion #first match
    return default #nothing derived

#classify will return class value only and not the prefix class=
def classify(r): #this classifies the car based on the relevant rules and returns the relevant class value only
    return predict(r, rules).split("=")[1] #"class=unacc" --> "unacc". comares with to come r["class"]

#predict most common class learned from training (baseline accuracy) and test score
majority = Counter(r["class"] for r in train).most_common(1)[0][0] #how many training cars are of majority class
baseline = sum(r["class"] == majority for r in test) /len(test) #how many test cares are of majority
print(f"Always predict {majority!r}: {baseline:.2f}") #baseline accuracy

y_true = [r["class"] for r in test] #real class for test car
y_pred = [classify(r) for r in test] #what rules predicted for each

#accuracy: correct predictions/all preds
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2f}")

#precision, recall, f1-score printed for each class
#precision: of all cars printed as x, how many are x?
#recall: of all cars that are really x, how many were foind?
print(classification_report(y_true, y_pred, zero_division=0)) #zero+division=0 just avoids warning if lcass is never predicted

#confus matrix: rows = real class, columns = predicted
#is in alphabetical (acc, good, unacc, vgood)
print(confusion_matrix(y_true, y_pred, labels=sorted(set(y_true)))) #rows = real class, colums = predicted class.
#labels = sorted(set(y_true)) all classes in test data. labels for confusion matrix

#pringing three cars to see wrong rules and show what is missig (for self evaluation purposes)
wrong = [r for r in test if classify(r) != r["class"]] #list misclassified cars. classify(r),what rules predicted. r["class"], real class
for r in wrong[:3]: #prints first three misclassified cars 
    print(r, "-->", classify(r)) #prints the misclassified car /rules predicted for it. r, misclassified car

#interactove
options = {
    "buying": ["vhigh", "high", "med", "low"], #[] providing the list of choices in a group not seperate
    "maint": ["vhigh", "high", "med", "low"],
    "doors": ["2", "3", "4", "5more"],
    "persons": ["2", "4", "more"],
    "lug_boot": ["small", "med", "big"],
    "safety": ["low", "med", "high"]
}

car = {}
for attribute, values in options.items(): # loop ask for each attribute
    answer = input(f"{attribute} {values}: ").strip().lower() #ignore space/capital letter
    while answer not in values: #validity checking
        answer = input (f"option invalid. please select from {values}: ").strip().lower()
    car[attribute] = answer #collect answer into dictionary for prediciton rate

print("Rating:", predict(car, rules)) #print based on earlier defined rules 

#explain the ratings coming rules fired for the car
known, trace = forward_chain_trace({f"{a}={v}" for a, v in car.items()}, rules)
for conditions, conclusion in trace: #show each rule fired
    print(f"{' + '.join(conditions)} --> {conclusion}")


In [ ]:
#backward chaining: top down approach

import csv 
import random
from collections import Counter

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
with open("car.data") as f:
    rows = [dict(zip(columns, line)) for line in csv.reader(f) if line] #dict(zip(columns, line)): make dictionary for each row with column name as keys and row value as values

random.seed(71)
shuffled = rows[:]
random.shuffle(shuffled)

cut = int(0.7 * len(shuffled)) 
train, test = shuffled[:cut], shuffled[cut:]

random.seed(71)#fixed seed so same split each run, results thereby reproducible. shuffling both groups gets a mix of classes
shuffled = rows[:] #copy so OG order remains
random.shuffle(shuffled) #mixy mix cars groups to get fair combination.else test set would only have a certain group of cars since car.data is sorted

cut = int(0.7 * len(shuffled)) #70% of 1728, 1209 = training data to design rules. the 519 test cars used to score the rules
train, test = shuffled[:cut], shuffled[cut:]

rules = [

    ({"safety=low"}, "class=unacc"), #not family safe or safe for daily usage
    ({"persons=2"}, "class=unacc"), #not gonna hold a family
    ({"buying=vhigh", "maint=vhigh"}, "class=unacc"),#expensive to buy AND run
    ({"buying=vhigh", "maint=high"}, "class=unacc"), # high buying ALONE isn't grounds for rejection
    ({"buying=high", "maint=vhigh"}, "class=unacc"),

    #vgood rules - ideal family car is safe with big boot and affordable
    ({"buying=low", "safety=high", "lug_boot=big"}, "class=vgood"), 
    ({"buying=med", "maint=med", "safety=high", "lug_boot=big"}, "class=vgood"),
    ({"buying=med", "maint=low", "safety=high", "lug_boot=big"}, "class=vgood"),

    #cheap buying and running: good
    ({"buying=low", "maint=low"}, "class=good"), 
    ({"buying=low", "maint=med"}, "class=good"),
    ({"buying=med", "maint=low"}, "class=good"),

    #reasoably safe with boot space. last since they're more broad. acc
    ({"safety=high"}, "class=acc"),
    ({"safety=med", "lug_boot=big"}, "class=acc"),
    ({"safety=med", "lug_boot=med"}, "class=acc")

]

def prove(goal, facts, rules, show=False, depth=0): #starts from goal e.g. 'vgood' and works back trying to PROVE each condition of any rule resulting in that goal. 
    #show=false won't print, runs in back and just gives answer of true for proved, false for not proved. used in predict for test cars only requiring yes/no
    #shpw = true prints each step for reasoning so you can see why it concluded what it did. 
    indent = "  " * depth #indents more in depth steps so proof is easier to comprehend
    if goal in facts: #goal previously a known fact, thus proved
        if show: print(f"{indent}{goal}: known fact")
        return True
    for conditions, conclusion in rules:
        if conclusion == goal: #cld prove goal
            if show: print(f"{indent}{goal}: try rule needing {sorted(conditions)}") #sorted tells it to try conditions in the same order each time
            #recursive function below to prove each condition one level deeper each time
            if all(prove(c, facts, rules, show, depth +1) for c in sorted(conditions)):
                if show: print(f"{indent}{goal}: proved") #prints whenever goal is proven
                return True

    #comes to play after eahc rule for the goal hasa been triled and failed
    if show: print(f"{indent}{goal}: cannot be proved")
    return False #no rule provable. 

#prediction proving the class goals in order - prove classes in order where first proved wins: priority order
goals = ["class=unacc", "class=vgood", "class=good", "class=acc"]

def predict(row, rules, default="class=unacc"): #predict function calls on backward_chaining
    #gotta leave out class column or the answer comes as known fact
    facts = {f"{a}={v}" for a, v in row.items() if a != "class"}
    for goal in goals: #priority order for classes
        if prove(goal, facts, rules):
            return goal #first goal proved comes out on top
    return default #default for if nothing is proved

def classify(r): #returns class value only instead of "class=" prefix. can be compare to real answer in r["class"]
    return predict(r, rules).split("=")[1]
